# HCCL 算子介绍

本节把「通信算子」这一层展开：先认全 8 个常见集合通信算子，
再看它们在四类并行方式里分别扮演什么角色。

## 本节学习大纲

- 常见集合通信算子总览
- 多对一与一对多：四个带 root 的算子
- 多对多：四个无 root 的算子
- 大模型的典型通信模型
- 课堂小练
- 本节总结

## 1. 常见集合通信算子总览

集合通信算子可按参与者与数据流向分为两大家族：**多对一 / 一对多**（存在 root 中心成员）
与 **多对多**（所有成员地位对等，没有 root）。下面这张图把 8 个常见算子放在一起对照。

![图 1](./images/png/collective_ops.png)

*图 1 · 八个常见集合通信算子：每张卡片左边是「操作前」、右边是「操作后」，看每个 Rank 手上分别有什么（左列：多对一 / 一对多；右列：多对多）*

观察这张图，可以抓住两个关键规律：

- **左列都有 root**：Gather 把各家的数据**汇集**到 root；Reduce 在汇集之后**再做一次归约**，
  4 份合成 1 份；Broadcast 把 root 的**整份数据原样复制**给每个人（人人一样）；Scatter 则把
  root 的整份数据**拆开**，每人只拿一片。
- **右列都没有 root**：AllReduce / AllGather / ReduceScatter / AlltoAll 的「All」都意味着
  **每个成员都要参与、每个成员都拿到结果**，区别在于拿到的是「同一份完整结果」还是「各自的分片」。

## 2. 多对一与一对多：四个带 root 的算子

| 算子 | 方向 | 语义 | 结果 |
| --- | --- | --- | --- |
| **Gather** | 多 → 一 | 收集通信域内所有成员的数据到 root | root 收到所有成员的完整数据 |
| **Reduce** | 多 → 一 | 收集所有成员数据到 root **并做 Reduce 归约** | root 收到归约后的结果 |
| **Broadcast** | 一 → 多 | 将 root 的数据**广播**到所有成员 | 所有成员都是 root 的数据 |
| **Scatter** | 一 → 多 | 将 root 的数据**均分**至所有成员 | 各成员各拿到一份分片 |

## 3. 多对多：四个无 root 的算子

| 算子 | 语义 | 结果 |
| --- | --- | --- |
| **AllReduce** | 每个成员收集所有成员的数据并做 Reduce | **所有人得到相同的归约结果** |
| **AllGather** | 每个成员收集所有成员的数据（不做归约） | **所有人得到完整的全体数据** |
| **ReduceScatter** | 先做 Reduce，再把结果**按维度均分**给各成员 | 每个成员得到结果的一个分片 |
| **AlltoAll** | 所有成员相互之间 scatter 与 gather 等量数据 | 每个成员收发等量数据 |

其中 **AllReduce 与 All2All 是最核心的两个算子**：

- **AllReduce** 是数据并行（DP）与张量并行（TP）的通信基础——梯度 / 激活值的归约与再分发；
- **All2All** 是专家并行（EP）的通信基础——token 的 **Dispatch（分发）与 Combine（回收）**。

> 一个有趣的观察：**AllReduce = ReduceScatter + AllGather**。
> 很多高性能实现正是把 AllReduce 拆成这两步来做，以降低单步的数据量压力。

### 3.1 五个最容易混的地方

| 容易混的一对 | 区别 |
| --- | --- |
| **Reduce vs Gather** | 都在 root 收数据。Reduce **多做了一次归约**（4 份 → 1 份）；Gather 只是把 4 份原样堆在 root |
| **Broadcast vs Scatter** | 都是「一 → 多」。Broadcast 发的是**同一份完整数据**（人人一样）；Scatter 把数据**拆开**，每人只拿一片 |
| **Gather vs AllGather** | Gather 只有 root 收到全部数据；AllGather 是**人人都**收到全部数据 |
| **Scatter vs AlltoAll** | 都涉及「分片」。Scatter 只有 root 在发；AlltoAll 是所有人**既发又收** |
| **ReduceScatter vs AllReduce** | ReduceScatter 把归约结果**切开**分给各成员；AllReduce 把**完整的**归约结果给每个成员 |

## 4. 大模型的典型通信模型

不同并行方式切分模型的方式不同，需要的通信算子、以及通信能否被计算「隐藏」（overlap）
也大不一样。下面这张图把四类并行的切分方式、通信算子与特征数据放在一起对照。

（四类并行**各自切的是什么**，见第 1 章 1.2 节 §3；本节重点看**用什么算子、通信量多大、能否隐藏**。）

![图 2](./images/png/large_model_comm.png)

*图 2 · 大模型典型通信模型：四类并行的切分方式与通信特征*

### 4.1 什么叫「把通信隐藏起来」

上面那张表里反复出现「能否隐藏」这个词。它的意思是：

> **让通信和计算同时进行**——这样通信耗时就「藏」在计算时间里，不额外占时间。

下面用**同样多的计算和同样多的通信**做对比，一眼就能看出区别：

![图 3](./images/png/overlap_timeline.png)

*图 3 · 能隐藏（数据并行）与不能隐藏（专家并行）的时间轴对比*

那为什么有的藏得住、有的藏不住？**关键看通信和计算之间有没有依赖**：

| 并行方式 | 能不能藏 | 为什么 |
| --- | --- | --- |
| **数据并行 DP** | ✅ 能 | 反向传播是**逐层**算的：某一层的梯度一算完就能发起 AllReduce，同时继续算前面层的反向——通信和计算并排跑 |
| **张量并行 TP** | ⚠️ 能藏一部分 | 被切开的是矩阵乘（Matmul）。某一层的 AllReduce 可以和**下一层**的矩阵乘重叠，但层内本身有先后依赖，藏不干净 |
| **流水并行 PP** | ✅ 能 | 不同 stage 本来就在流水作业，通信藏在流水里 |
| **专家并行 EP** | ❌ 不能 | 必须**先** All2All 把 token 分发到专家所在的卡，专家**才能**开始算；算完还要 All2All 把结果收回来，下一层才能继续。通信和计算是**串行**的 |

> 所以「能不能隐藏」不是看通信量大不大，而是看**通信和计算之间有没有依赖**——有依赖就只能排队。

### 4.2 三条结论

1. **数据量差距悬殊**：DP 的梯度同步单次可达 **GB 级**，PP 只是十 MB~百 MB 量级；
   这决定了它们对带宽与算法的诉求完全不同。
2. **能否隐藏差别很大**：DP 和 PP 能把通信藏住，TP 能藏一部分；
   而 **EP 的计算与通信必须串行、藏不住**，这也是 MoE 训练最需要优化通信的原因。
3. **节点规模**：DP 可扩展到 **1000+ 卡**，EP 到 256 卡，TP 到 64 卡——规模越大，
   通信算法的选择（Ring / Mesh / Halving-Doubling）对性能的影响越显著。

## 5. 课堂小练

请根据本节学习内容完成以下题目进行自测。

**1、（连线题）常见的集合通信算子**

请把左侧描述与右侧算子配对：

| 描述 | 算子 |
| --- | --- |
| 按维度执行归约并发送到对应 NPU 上 | A. AllGather |
| 收集每个 NPU 所有的数据到每个 NPU 上 | B. AllReduce |
| 所有 NPU 互相 scatter 和 gather 数据 | C. All2All |
| 从多个 NPU 获取数据到所有 NPU 并做归约运算 | D. ReduceScatter |

---

**2、（单选题）** 稀疏 MoE 模型分布式训练中，「将数据分发给对应专家、从对应专家接收计算后的结果」
这一 Dispatch-Combine 流程，使用到的集合通信算子类别是？

- **A)** AllReduce 算子
- **B)** AllGather 类算子
- **C)** AlltoAll 类算子
- **D)** ReduceScatter 类算子

---

**3、（单选题）** 下列关于四类并行方式与其通信算子的对应，**错误**的是？

- **A)** 数据并行（DP）使用 allreduce 做梯度同步
- **B)** 流水并行（PP）使用 Send / Recv 传递层间激活值
- **C)** 张量并行（TP）使用 all2all 完成层内结果的拼回
- **D)** 专家并行（EP）使用 all2all 完成 token 的分发与回收

---

**4、（判断题）** Broadcast 与 Scatter 的区别只是「发的人多少」。

---

**5、（单选题）** 下列哪个算子让**每个成员**都拿到**同一份归约结果**？

- **A)** AllGather
- **B)** Reduce
- **C)** AllReduce
- **D)** ReduceScatter

---

**6、（判断题）** 专家并行（EP）的通信能被计算隐藏，所以对训练效率影响很小。

**执行以下代码获取答案。**

In [ ]:
# 读取本节课堂小练答案（用 Python 读文件：跨平台，并保证中文不乱码）
import pathlib
_p = pathlib.Path('./answer/02.02_answer.txt')
if _p.exists():
    print(_p.read_text(encoding='utf-8'))
else:
    print(f'未找到答案文件：{_p}')
    print('（请确认 answer/ 目录与本 notebook 在同一层）')

## 6. 本节总结

### 八个常见集合通信算子

- **多对一 / 一对多**（有 root）：Gather、Reduce、Broadcast、Scatter；
- **多对多**（无 root）：AllReduce、AllGather、ReduceScatter、AlltoAll；
- **最核心的两个**：AllReduce（DP / TP 的基础）与 All2All（EP 的基础）。

### 大模型典型通信模型

| 并行类型 | 操作类型 | 节点规模 | 数据量 | 能否隐藏 |
| --- | --- | --- | --- | --- |
| 张量并行 TP | allreduce / allgather / reducescatter | Max to 64 Chip | 百 MB ~ GB | 可隐藏部分 |
| 流水并行 PP | Send/Recv | 2 Chip | 十 MB ~ 百 MB | 可流水隐藏 |
| 数据并行 DP | allreduce | Max to 1000+ Chip | GB | 可 overlap 隐藏 |
| 专家并行 EP | all2all | Max to 256 Chip | MB ~ 百 MB | **串行，不可隐藏** |